In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load your data
df = pd.read_csv('heart_failure_data.csv')

# Target and features
y = df['HF'].apply(lambda x: 1 if x in [1, 2] else 0)  # binary target: 0 vs 1/2

# Feature sets
X_all = df[['EF', 'GLS', 'QRS']]      # for SVM and RF
X_no_gls = df[['EF', 'QRS']]          # for XGBoost

# Scale features
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

scaler_nogls = StandardScaler()
X_no_gls_scaled = scaler_nogls.fit_transform(X_no_gls)

# Train/test split
X_train_all, X_test_all, y_train, y_test = train_test_split(X_all_scaled, y, test_size=0.2, random_state=42, stratify=y)
X_train_nogls, X_test_nogls, _, _ = train_test_split(X_no_gls_scaled, y, test_size=0.2, random_state=42, stratify=y)

# Define models
svm_model = SVC(kernel='rbf', C=10, probability=False, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=42)
xgb_model = XGBClassifier(eval_metric='logloss', colsample_bytree=0.8, learning_rate=0.01,
                          max_depth=3, n_estimators=50, subsample=0.8, random_state=42)

# Train each model
svm_model.fit(X_train_all, y_train)
rf_model.fit(X_train_all, y_train)
xgb_model.fit(X_train_nogls, y_train)

# Custom hard voting
svm_pred = svm_model.predict(X_test_all)
rf_pred = rf_model.predict(X_test_all)
xgb_pred = xgb_model.predict(X_test_nogls)

# Majority voting
pred_stack = np.vstack((svm_pred, rf_pred, xgb_pred))
y_pred = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=pred_stack)

# Evaluate
print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Voting Classifier Accuracy: 0.9583333333333334
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.92      0.96        12
           1       0.92      1.00      0.96        12

    accuracy                           0.96        24
   macro avg       0.96      0.96      0.96        24
weighted avg       0.96      0.96      0.96        24

Confusion Matrix:
 [[11  1]
 [ 0 12]]
